# Question 2: Do winning teams have higher possession than losing teams?

## Analytic question
This analysis compares the average possession percentage of winning and losing teams in completed matches. Draws are excluded because they have no winning or losing team.

## Hypotheses
- **H0:** The mean possession of winners is equal to the mean possession of losers.
- **H1:** The mean possession of winners is higher than the mean possession of losers.

Each match contributes one winner and one loser, so the final test uses the paired possession difference within each match. This avoids treating the two observations from the same match as independent.

## Required input
Place a match-level CSV named `possession_matches.csv` in the `datasets` folder. It must contain these columns:
`HomeTeam`, `AwayTeam`, `FTHG`, `FTAG`, `HomePossession`, and `AwayPossession`.

The two possession columns must be percentages, such as `57` or `57%`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_PATH = "../datasets/possession_matches.csv"
ALPHA = 0.05

matches = pd.read_csv(DATA_PATH)
required_columns = {
    "HomeTeam", "AwayTeam", "FTHG", "FTAG",
    "HomePossession", "AwayPossession"
}
missing_columns = required_columns.difference(matches.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print("Raw matches:", len(matches))
display(matches.head())

## Data cleaning and match result
Scores and possession are converted to numeric values. Matches with missing or invalid values are removed before the comparison.

In [ ]:
numeric_columns = ["FTHG", "FTAG", "HomePossession", "AwayPossession"]
for column in numeric_columns:
    matches[column] = (
        matches[column].astype(str).str.replace("%", "", regex=False).str.strip()
    )
    matches[column] = pd.to_numeric(matches[column], errors="coerce")

matches = matches.dropna(subset=numeric_columns).copy()
matches = matches[
    matches["HomePossession"].between(0, 100)
    & matches["AwayPossession"].between(0, 100)
].copy()

matches["Result"] = np.select(
    [matches["FTHG"] > matches["FTAG"], matches["FTHG"] < matches["FTAG"]],
    ["Home win", "Away win"],
    default="Draw"
)

completed = matches[matches["Result"] != "Draw"].copy()
winners = np.where(completed["Result"] == "Home win", completed["HomePossession"], completed["AwayPossession"])
losers = np.where(completed["Result"] == "Home win", completed["AwayPossession"], completed["HomePossession"])
comparison = pd.DataFrame({"Winner possession": winners, "Loser possession": losers})
comparison["Difference"] = comparison["Winner possession"] - comparison["Loser possession"]

print("Usable matches:", len(matches))
print("Draws excluded:", (matches["Result"] == "Draw").sum())
print("Winner-loser pairs:", len(comparison))
display(comparison.head())

## Descriptive statistics and 95% confidence interval
The confidence interval is for the mean within-match possession difference: winner possession minus loser possession.

In [ ]:
summary = comparison[["Winner possession", "Loser possession", "Difference"]].agg(["count", "mean", "median", "std", "min", "max"]).T
display(summary.round(2))

differences = comparison["Difference"]
mean_difference = differences.mean()
standard_error = stats.sem(differences)
confidence_interval = stats.t.interval(
    0.95, df=len(differences) - 1, loc=mean_difference, scale=standard_error
)

print(f"Mean winner-minus-loser difference: {mean_difference:.2f} percentage points")
print(f"95% confidence interval: {confidence_interval[0]:.2f} to {confidence_interval[1]:.2f} percentage points")

## Paired hypothesis test
A two-sided paired t-test is reported because it tests whether the average within-match possession difference differs from zero. The one-sided research question is supported when the estimated difference is positive and the two-sided p-value is below 0.05.

In [ ]:
test = stats.ttest_rel(comparison["Winner possession"], comparison["Loser possession"])

print(f"Paired t-statistic: {test.statistic:.3f}")
print(f"Two-sided p-value: {test.pvalue:.4f}")

if test.pvalue < ALPHA and mean_difference > 0:
    print("Conclusion: Winning teams had significantly higher average possession than losing teams.")
elif test.pvalue < ALPHA:
    print("Conclusion: Winning teams had significantly lower average possession than losing teams.")
else:
    print("Conclusion: The sample does not provide statistically significant evidence that winners had higher possession than losers.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(
    [comparison["Winner possession"], comparison["Loser possession"]],
    labels=["Winning teams", "Losing teams"]
)
plt.ylabel("Possession (%)")
plt.title("Possession of winning and losing teams")
plt.grid(axis="y", alpha=0.25)
plt.show()

## Interpretation
The mean difference, confidence interval, p-value, and boxplot should be reported together. A statistically significant result indicates an association in this sample, not that possession alone causes a team to win.